# EDA 13jul2026
### Imports necesarios para el EDA

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from scipy.stats import chi2_contingency
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
from dotenv import load_dotenv

**NOTA**: Crear .env con variables MYSQL_PASSWORD, MYSQL_HOST, MYSQL_PORT, MYSQL_USER, MYSQL_DATABASE dentro de la propia carpeta SCRIPTS para reejectuar EDA.

El .env está incluido en .gitignore y no se copiará en el repositorio

### Conexión a la BBDD
 

In [ ]:
load_dotenv(dotenv_path=".env") 
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = os.getenv("MYSQL_PORT")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE")

In [ ]:
# Crear motor de conexión
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

### Carga dataset
**NOTA**: cambiar nombre tabla para reejecutar semanalmente

In [ ]:
table = "RRHH_CLEAN_13072026"

In [ ]:
with engine.connect():
    query = f"SELECT * FROM {table};"
    df = pd.read_sql(query, engine)

### Dimensiones dataset

In [ ]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

### Muestra del dataset

In [ ]:
# Muestra de las cinco primeras filas del DataFrame
df.head()

In [ ]:
# Muestra de cinco filas aleatorias del DataFrame
df.sample(5, random_state=42) # random_state muestra mismas filas ejecutadas x primera vez

### Tipos de datos

In [ ]:
tipos_datos = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "valores_no_nulos": df.notna().sum(),
    "valores_nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2)
}).reset_index().rename(columns={"index": "columna"})

tipos_datos

### Duplicados

In [ ]:
duplicados_detalle = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print(f"Registros involucrados en duplicados completos: {len(duplicados_detalle)}")
duplicados_detalle.head(10)

In [ ]:
duplicados = df.duplicated().sum()

print(f"Filas duplicadas: {duplicados}")
print(f"Porcentaje duplicado: {duplicados / len(df) * 100:.2f}%")


### Clasificación de variables en categórica y númericas ( discretas y continuas )

In [ ]:
variables_categoricas = [
    'Reason_Absence', 'Month_Absence', 'Day_Week', 'Seasons', 'Education'
]

variables_booleanas = ['Disciplinary_Failure', 'Social_Drinker', 'Social_Smoker']

variables_discretas = ['Son', 'Pet' ]

variables_continuas = [
    'Transportation_Expense', 'Distance_Residence_Work', 'Work_load_Average_Day', 'Age', 
    'Hit_Target', 'Weight', 'Height', 'Body_Mass_Index', 'Absenteeism_Hours', 'Service_Time'
]


### Conversiones de datos para facilitar el EDA

In [ ]:
# Convertir las columnas "Son" y "Pet" a tipo int para que puedan ser utilizadas como numéricas.

df['Son'] = df['Son'].astype(int)
df['Pet'] = df['Pet'].astype(int)


# Convertir la columna "Work_load_Average_Day" a tipo float para que pueda ser utilizada como númerica.

df["Work_load_Average_Day"] = (
    df["Work_load_Average_Day"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# Convertir las columnas categóricas a tipo "category".

for col in variables_categoricas:
    df[col] = df[col].astype("category")

### Analisis Estadístico 

#### Variables númericas

In [ ]:
df.columns

In [ ]:
variables_numericas = variables_discretas + variables_continuas

df[variables_numericas].describe().T.round(2)

#### Variables categóricas

In [ ]:
df[variables_categoricas+variables_booleanas].describe().T

### Visualizaciones

In [ ]:
sns.set_theme(style="whitegrid")

#### Distribución Numéricas Continuas

In [ ]:
for col in variables_continuas:
    plt.figure(figsize=(7, 4))
    ax =sns.histplot(data=df, x=col, bins=30)

    ax.bar_label(ax.containers[0], padding=1, fontsize=8)

    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

#### Distribución númericas discretas

In [ ]:
for col in variables_discretas:
    tabla_frecuencia = (
        df[col]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis(col)
        .reset_index(name="frecuencia")
    )
    
    tabla_frecuencia["porcentaje"] = (
        tabla_frecuencia["frecuencia"] / len(df) * 100
    ).round(2)
    
    display(tabla_frecuencia)
    
    plt.figure(figsize=(7, 4))
    ax = sns.barplot(data=tabla_frecuencia, x=col, y="frecuencia")
    for container in ax.containers:
        ax.bar_label(container, fmt="%d", padding=3)

    plt.title(f"Frecuencia de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()

#### Detección de Outliers (continuas)

##### Resumen cuantitativo de outliers

In [ ]:
resumen_outliers = []

for col in variables_continuas:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    mascara = (df[col] < limite_inferior) | (df[col] > limite_superior)

    resumen_outliers.append({
        "variable": col,
        "limite_inferior": round(limite_inferior, 2),
        "limite_superior": round(limite_superior, 2),
        "n_outliers": int(mascara.sum()),
        "porcentaje_outliers": round(mascara.mean() * 100, 2),
        "min": df[col].min(),
        "max": df[col].max()
    })

resumen_outliers = pd.DataFrame(resumen_outliers).sort_values("porcentaje_outliers", ascending=False)
resumen_outliers


In [ ]:
for col in variables_continuas:
    plt.figure(figsize=(7, 3))
    sns.boxplot(data=df, x=col)
    plt.title(f"Boxplot de {col}")
    plt.xlabel(col)
    plt.show()

#### Frecuencias Categóricas

In [ ]:
ordenes_categoricas = {
    "Month_Absence": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "Day_Week": [2, 3, 4, 5, 6],
    "Seasons": [1, 2, 3, 4],
    "Education": [1, 2, 3, 4]

}

diccionario_categoricas = {
    "Month_Absence": {
        0:"Sin mes",
        1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr",
        5: "May", 6: "Jun", 7: "Jul", 8: "Ago",
        9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"
    },
    "Day_Week": {
        2: "Lun", 3: "Mar", 4: "Mié", 5: "Jue", 6: "Vie"
    },
    "Seasons": {
        1: "Invierno", 2: "Primavera", 3: "Verano", 4: "Otoño"
    },
    "Education": {
        1: "Secundaria", 2: "Graduado", 3: "Postgrado", 4: "Master o doctorado"
    }

}

In [ ]:
for col in variables_categoricas:
    plt.figure(figsize=(8, 4))
    orden = df[col].value_counts().index
    ax =sns.countplot(data=df, x=col, order=orden)
    for container in ax.containers:
        ax.bar_label(container, fmt="%d", padding=3)

    if col in diccionario_categoricas:
        labels = [diccionario_categoricas[col].get(x, x) for x in orden]
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels)

    plt.title(f"Frecuencia de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=0)
    plt.show()

#### Porcentaje Booleanas

In [ ]:
for col in variables_booleanas:
    serie = pd.to_numeric(df[col], errors="coerce")
    tabla = (
        serie
        .map({0: "No", 1: "Sí"})
        .value_counts(normalize=True)
        .mul(100)
        .reindex(["No", "Sí"])
        .reset_index()
    )

    tabla.columns = [col, "porcentaje"]

    ax = sns.barplot(data=tabla, x=col, y="porcentaje")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=3)

    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Porcentaje")
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()

#### Relaciones entre variables

##### Gráficos de dispersión

In [ ]:
pares_dispersion = [
    ("Age", "Service_Time"),
    ("Distance_Residence_Work", "Transportation_Expense"),
    ("Work_load_Average_Day", "Absenteeism_Hours"),
    ("Distance_Residence_Work", "Absenteeism_Hours"),
    ("Hit_Target", "Absenteeism_Hours"),
    ("Age", "Absenteeism_Hours"),
    ("Service_Time", "Absenteeism_Hours")
]

for x, y in pares_dispersion:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=df, x=x, y=y, alpha=0.6)
    plt.title(f"{x} vs {y}")
    plt.xlabel(x)
    plt.ylabel(y)
    plt.show()

##### Matriz de correlación

In [ ]:
variables_correlacion = variables_discretas + variables_continuas

matriz_correlacion = df[variables_correlacion].corr(method="spearman")

plt.figure(figsize=(10, 8))
sns.heatmap(
    matriz_correlacion,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    cbar=False
)
plt.title("Matriz de correlación")
plt.show()

##### Cruce de categóricas


In [ ]:
pares_categoricos = [
    ("Social_Drinker", "Social_Smoker"),
    ("Education", "Disciplinary_Failure"),
    ("Day_Week", "Reason_Absence"),
    ("Seasons", "Reason_Absence")
]

for col1, col2 in pares_categoricos:
    tabla_porcentaje = pd.crosstab(
        df[col1],
        df[col2],
        normalize="index"
    ) * 100

    plt.figure(figsize=(9, 5))
    sns.heatmap(
        tabla_porcentaje,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        cbar=False
    )

    plt.title(f"Distribución porcentual de {col2} según {col1}")
    plt.xlabel(col2)
    plt.ylabel(col1)
    plt.tight_layout()
    plt.show()

In [ ]:
def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabla)[0]
    n = tabla.sum().sum()
    r, k = tabla.shape

    return np.sqrt((chi2 / n) / min(k - 1, r - 1))

asociaciones_categoricas = []

variables_asociaciones = variables_categoricas + variables_booleanas

for i, col1 in enumerate(variables_asociaciones):
    for col2 in variables_asociaciones[i+1:]:
        asociaciones_categoricas.append({
            "variable_1": col1,
            "variable_2": col2,
            "cramers_v": cramers_v(df[col1], df[col2])
        })

asociaciones_categoricas = pd.DataFrame(asociaciones_categoricas)

asociaciones_categoricas.sort_values("cramers_v", ascending=False)

##### Cruce categóricas, numéricas

In [ ]:
cruces = [
    ("Education", "Hit_Target"),
    ("Education", "Service_Time"),
    ("Education", "Absenteeism_Hours"),
    ("Disciplinary_Failure", "Absenteeism_Hours"),
    ("Disciplinary_Failure", "Hit_Target"),
    ("Social_Drinker", "Body_Mass_Index"),
    ("Social_Smoker", "Absenteeism_Hours")
]

for cat, num in cruces:
    display(
        df.groupby(cat)[num]
        .agg(["count", "mean", "median", "std"])
        .round(2)
    )

    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x=cat, y=num)
    plt.title(f"{num} según {cat}")
    plt.tight_layout()
    plt.show()

### Conclusiones

#### Conclusiones Generales

Se mantiene la validez de las conclusiones del resumen 2026-06-29_EDA.
Hay más registros pero todas las tendencias se mantienen.

Resultados una vez se han eliminado los duplicados y vuelto a ejecutar:

- Total registros: 845 (+105 nuevos)  
- Total: 136 empleados (+100 nuevos)  
- Sin nulos  
- Sigue apareciendo month=0  
- % duplicados se mantiene (4,62% vs 4,59%)  
- Variables numéricas mantienen min y max.  
- Variables categóricas no han sumados categorias nuevas, y frecuencia top se mantiene en cada una.  
- Las distribuciones de todas las variables se mantienen. De hecho, sólo se suman registros a los valores existentes manteniendo la frecuencia general de cada valor.  
- Work_load_Average_day tiene alguns cambios con sus nuevos registros.  



<!-- #### Conclusiones Relaciones

Correlaciones fuertes o relevantes
* Weight y Body_Mass_Index: 0.89
Normal dado que el IMC depende directamente del peso.

* Age y Service_Time: 0.76
Empleados de mayor edad tienden a tener más tiempo de servicio.

    * Body_Mass_Index y Service_time: 0.56

    * Age y Body_mass_index: 0.48

    * Weight y Service_time: 0.53

* Son y Transportation_Expense: 0.42
* Pet y Transportation_Expense: 0.47
Empleado con hijos o mascotas tienden a gastar más en transporte.


Relaciones negativas
* Pet y Service_time: -0.34
Empleados con más tiempo de servicio tienden a tener menos mascotas. 
* Pet y Age: -0.28
Empleados con más edad tienden a tener menos mascotas. 

* Transportation_expense y Service_time: -0.28
A mayor tiempo de servicio, menor gasto de transporte. Podría deberse a perfiles de empleados, ubicación o antigüedad.


Cruces de categóricas

* Reason_absence vs Disciplinary_Failure: 0.962
 Probablemente porque ciertos motivos de ausencia están directamente ligados a fallo disciplinario o ausencia injustificada. Revisaría la tabla cruzada: puede haber una categoría casi exclusiva.

* Education vs Social_Drinker: 0.472
Asociación moderada. El consumo social de alcohol varía según nivel educativo.

* Education vs Social_Smoker: 0.364
Asociación moderada-baja. El hábito de fumar también cambia según educación, aunque menos que beber.

Estas sugieren que los motivos de ausencia no se distribuyen igual según hábitos sociales o estación:
 
* Reason_absence vs Social_Smoker: 0.356
* Reason_absence vs Seasons: 0.347
* Reason_absence vs Social_Drinker: 0.346 -->
